In [2]:

import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from pathlib import Path
import shutil
import torch
from ultralytics import YOLO


In [11]:
if torch.cuda.is_available():
    print("GPU             :", torch.cuda.get_device_name(0))
    device = "0"
else:
    print("WARNING: GPU not detected. Using CPU.")
    device = "cpu"

GPU             : NVIDIA GeForce RTX 4050 Laptop GPU


In [4]:

dataset_path = Path("Traffic Violations Dataset")

train_path = dataset_path / "train"
val_path = dataset_path / "validation"
test_path = dataset_path / "test"

print("\nDataset:")
print("Train      :", train_path)
print("Validation :", val_path)
print("Test       :", test_path)


Dataset:
Train      : Traffic Violations Dataset\train
Validation : Traffic Violations Dataset\validation
Test       : Traffic Violations Dataset\test


In [5]:
if not train_path.exists():
    raise FileNotFoundError(f"Training folder not found: {train_path}")

if not val_path.exists():
    raise FileNotFoundError(f"Validation folder not found: {val_path}")

if not test_path.exists():
    raise FileNotFoundError(f"Test folder not found: {test_path}")


In [7]:
classes = sorted([
    folder.name
    for folder in train_path.iterdir()
    if folder.is_dir()
])

print("\nClasses:")
for i, cls in enumerate(classes):
    print(f"{i} -> {cls}")



Classes:
0 -> blur
1 -> helmet
2 -> no_helmet
3 -> overloading


In [8]:
print("\nLoading YOLO model...")

model = YOLO("yolo11n-cls.pt")

print("YOLO model loaded successfully.")



Loading YOLO model...
YOLO model loaded successfully.


In [9]:
def print_epoch_accuracy(trainer):

    epoch = trainer.epoch + 1
    total_epochs = trainer.epochs

    metrics = trainer.metrics

    top1 = metrics.get("metrics/accuracy_top1", None)
    top5 = metrics.get("metrics/accuracy_top5", None)

    print("\n" + "-" * 60)
    print(f"EPOCH {epoch}/{total_epochs} COMPLETED")

    if top1 is not None:
        print(f"Top-1 Accuracy : {top1 * 100:.2f}%")

    if top5 is not None:
        print(f"Top-5 Accuracy : {top5 * 100:.2f}%")

    print("-" * 60)


model.add_callback(
    "on_fit_epoch_end",
    print_epoch_accuracy
)


In [12]:
print("\n" + "=" * 60)
print("STARTING YOLO TRAINING")
print("=" * 60)

results = model.train(

    data=str(dataset_path),

    epochs=3,

    batch=32,

    imgsz=224,

    device=device,

    workers=4,

    project="Traffic_Violation_Runs",

    name="helmet_classification",

    save=True,

    val=True,

    patience=10,

    cache=False,

    verbose=True
)



STARTING YOLO TRAINING
New https://pypi.org/project/ultralytics/8.4.161 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.120  Python-3.13.9 torch-2.13.0+cu132 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=Traffic Violations Dataset, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=3, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_

In [15]:
print("\n" + "=" * 60)
print("TRAINING COMPLETED")
print("=" * 60)

best_model_path = (
    Path("Traffic_Violation_Runs")
    / "helmet_classification"
    / "weights"
    / "best.pt"
)

print("Best model:")
print(best_model_path)


TRAINING COMPLETED
Best model:
Traffic_Violation_Runs\helmet_classification\weights\best.pt


In [17]:
if not best_model_path.exists():

    print("\n❌ best.pt was not found at:")
    print(best_model_path)

    print("\nSearching for best.pt...")

    found_models = list(Path(".").rglob("best.pt"))

    if found_models:
        print("\nFound these models:")

        for i, path in enumerate(found_models):
            print(f"{i}: {path}")

        # Use the first found model
        best_model_path = found_models[0]

        print("\nUsing:")
        print(best_model_path)

    else:
        raise FileNotFoundError(
            "No best.pt file was found. "
            "Make sure YOLO training completed successfully."
        )




❌ best.pt was not found at:
Traffic_Violation_Runs\helmet_classification\weights\best.pt

Searching for best.pt...

Found these models:
0: runs\classify\Traffic_Violation_Runs\helmet_classification\weights\best.pt

Using:
runs\classify\Traffic_Violation_Runs\helmet_classification\weights\best.pt


In [18]:
print("\n" + "=" * 60)
print("LOADING BEST MODEL")
print("=" * 60)

best_model = YOLO(str(best_model_path))

print("Model loaded successfully!")
print("Model path:", best_model_path)


LOADING BEST MODEL
Model loaded successfully!
Model path: runs\classify\Traffic_Violation_Runs\helmet_classification\weights\best.pt


In [19]:
print("\nClasses learned by model:")

for class_id, class_name in best_model.names.items():
    print(f"{class_id} -> {class_name}")



Classes learned by model:
0 -> blur
1 -> helmet
2 -> no_helmet
3 -> overloading


In [20]:
# ============================================================
# 11. TEST / VALIDATE MODEL
# ============================================================

print("\n" + "=" * 60)
print("TESTING BEST MODEL")
print("=" * 60)

metrics = best_model.val(
    data=str(dataset_path),
    split="test",
    imgsz=224,
    batch=32,
    device="0"
)


TESTING BEST MODEL
Ultralytics 8.4.120  Python-3.13.9 torch-2.13.0+cu132 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
YOLO11n-cls summary (fused): 47 layers, 1,531,148 parameters, 0 gradients, 3.2 GFLOPs
train: C:\My Space\Github_Repo\Traffic Violation Detection-Deep Learning\Traffic Violations Dataset\train... found 2396 images in 4 classes  
val: C:\My Space\Github_Repo\Traffic Violation Detection-Deep Learning\Traffic Violations Dataset\validation... found 402 images in 4 classes  
ERROR test: C:\My Space\Github_Repo\Traffic Violation Detection-Deep Learning\Traffic Violations Dataset\test... found 297 images in 3 classes (requires 4 classes, not 3)
WARNING test: Slow image access detected (ping: 0.50.5 ms, read: 26.68.2 MB/s, size: 12.8 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
test: Scanning C:\My Space\Github_Repo\Traffic Violation Detection-Deep Learning\Traffic Violation

In [21]:
# ============================================================
# 12. PRINT TEST ACCURACY
# ============================================================

print("\n" + "=" * 60)
print("FINAL TEST ACCURACY")
print("=" * 60)

print(f"Top-1 Accuracy : {metrics.top1 * 100:.2f}%")
print(f"Top-5 Accuracy : {metrics.top5 * 100:.2f}%")



FINAL TEST ACCURACY
Top-1 Accuracy : 81.48%
Top-5 Accuracy : 100.00%


In [22]:
# ============================================================
# 13. SAVE BEST MODEL TO saved_models
# ============================================================

saved_models = Path("saved_models")
saved_models.mkdir(parents=True, exist_ok=True)

final_model_path = saved_models / "helmet_model.pt"

shutil.copy(
    best_model_path,
    final_model_path
)

print("\n" + "=" * 60)
print("MODEL SAVED")
print("=" * 60)

print("Final model:")
print(final_model_path)


MODEL SAVED
Final model:
saved_models\helmet_model.pt
